# Generate Popularity-Enriched QA Datasets

This notebook processes three QA datasets (PopQA, Natural Questions, TriviaQA) and enriches them with Wikipedia popularity metrics.

**Output Files:**
- `popqa_with_popularity.parquet`
- `nq_with_popularity.parquet`
- `triviaqa_with_popularity.parquet`

**Output Schema:**
- `question_id`: Unique identifier for the question
- `question_text`: The question text
- `answer_texts`: List of answer strings
- `wikipedia_id`: Wikipedia page ID (used for matching across all datasets)
- `wikipedia_title`: Wikipedia page title
- `popularity_avg`: Average monthly pageviews (Jan 2022 - Jan 2023)
- `popularity_rank`: Popularity rank (from rank_avg in source dataset)

## 1. Imports and Configuration

In [19]:
import gc
import json
import os
import pandas as pd
from datasets import load_dataset
from config import DATA_DIR, CACHE_DIR, ROOT_DIR
from tqdm import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure output paths
OUTPUT_DIR = Path(DATA_DIR)
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

POPQA_OUTPUT = OUTPUT_DIR / "popqa_with_popularity.parquet"
NQ_OUTPUT = OUTPUT_DIR / "nq_with_popularity.parquet"
TRIVIAQA_OUTPUT = OUTPUT_DIR / "triviaqa_with_popularity.parquet"

# Dataset paths
HUGGINGFACE_POP_WIKI = "Cyro1/enwiki_pageviews_m"
POPQA_PATH = "akariasai/PopQA"
NQ_PATH = "facebook/kilt_tasks"
TRIVIAQA_PATH = "facebook/kilt_tasks"

# HuggingFace upload configuration
HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE_TOKEN")
HUGGINGFACE_UPLOAD_REPO = "Cyro1/popularity-enriched-qa-datasets"
HUGGINGFACE_UPLOAD_SUBSETS = {
    "PopQA": ("popqa", POPQA_OUTPUT),
    "NaturalQuestions": ("natural_questions", NQ_OUTPUT),
    "TriviaQA": ("triviaqa", TRIVIAQA_OUTPUT),
}

README_TEXT = """# Popularity-Enriched QA Datasets

This dataset repo hosts popularity-enriched versions of PopQA, Natural Questions, and TriviaQA.
Each subset retains the enrichment schema produced by this notebook (question + pron and popularity metrics).

## Subsets

- `popqa`: Popularity-enriched PopQA test split
- `natural_questions`: Wikipedia-provenance Natural Questions validation set
- `triviaqa`: TriviaQA validation subset matched to KILT and original TriviaQA IDs

## Schema

- `question_id`: question identifier
- `question_text`: raw question text
- `answer_texts`: list of candidate answers
- `wikipedia_id`: Wikipedia provenance page id
- `wikipedia_title`: page title
- `popularity_avg`: average monthly pageviews
- `popularity_rank`: rank derived from the popularity source

## Loading

Use `datasets.load_dataset("Cyro1/popularity-enriched-qa-datasets", split="popqa")` to stream the PopQA subset and swap `split` for each subset name.
"""

print("Configuration loaded successfully!")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Cache directory: {CACHE_DIR}")
print(f"HuggingFace repo: {HUGGINGFACE_UPLOAD_REPO}")
print("README text will be published directly to the HuggingFace dataset repo.")

Configuration loaded successfully!
Output directory: /Users/cyro/Documents/VSC/PopularityBias/data
Cache directory: /Users/cyro/Documents/VSC/PopularityBias/data/cache
HuggingFace repo: Cyro1/popularity-enriched-qa-datasets
README text will be published directly to the HuggingFace dataset repo.


## 2. Load Base Datasets

Load Wikipedia popularity data that will be joined with all QA datasets.

In [20]:
print("Loading Wikipedia popularity data...")
pop_ds = load_dataset(
    HUGGINGFACE_POP_WIKI,
    split="train+test",
    cache_dir=CACHE_DIR
)

# Convert to DataFrame and clean
pop_df = (
    pop_ds
    .select_columns(["wikipedia_id", "wikipedia_title", "popularity_avg", "rank_avg"])
    .to_pandas()
    .rename(columns={"rank_avg": "popularity_rank"})
)

# Clean titles
pop_df["wikipedia_title"] = pop_df["wikipedia_title"].str.strip()
pop_df = pop_df[pop_df["popularity_avg"].notna()].copy()
pop_df["wikipedia_id"] = pop_df["wikipedia_id"].astype("int64")

print(f"✓ Loaded {len(pop_df):,} Wikipedia pages with popularity data")
print(f"  Popularity range: {pop_df['popularity_avg'].min():.2f} - {pop_df['popularity_avg'].max():.2f}")
display(pop_df.head())

# Create ID lookup for faster matching
pop_lookup = pop_df.set_index("wikipedia_id")

Loading Wikipedia popularity data...
✓ Loaded 5,890,044 Wikipedia pages with popularity data
  Popularity range: 1.00 - 175763171.77


Loading Wikipedia popularity data...
✓ Loaded 5,890,044 Wikipedia pages with popularity data
  Popularity range: 1.00 - 175763171.77


,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank
0,11566748,Ossi Oikarinen,71.437500,2.957879e+06
1,37228152,2012–13 NBL Canada season,49.354167,3.259824e+06
2,30849763,Quercus iberica,93.708333,2.555272e+06
3,14734767,Fladnitz im Raabtal,12.791667,4.756272e+06
4,18662646,Jaszczołty,6.208333,5.296610e+06


## 3. Process PopQA Dataset

In [21]:
print("="*80)
print("PROCESSING POPQA DATASET")
print("="*80)

# Load PopQA
print("\nLoading PopQA dataset...")
popqa_ds = load_dataset(
    POPQA_PATH,
    split="test",
    cache_dir=CACHE_DIR
)

print(f"Loaded {len(popqa_ds):,} questions")
print("\nSample question:")
print(popqa_ds[0])

# Helper to normalize the possible answer payload
def normalize_possible_answers(raw_answers):
    if not raw_answers:
        return []
    if isinstance(raw_answers, str):
        try:
            raw_answers = json.loads(raw_answers)
        except json.JSONDecodeError:
            raw_answers = [raw_answers]
    if isinstance(raw_answers, dict):
        raw_answers = [raw_answers]
    if not isinstance(raw_answers, list):
        return []
    normalized = []
    for entry in raw_answers:
        if isinstance(entry, dict):
            candidate = entry.get("text") or entry.get("answer") or entry.get("value")
        elif isinstance(entry, (int, float)):
            candidate = str(entry)
        elif isinstance(entry, str):
            candidate = entry
        else:
            continue
        if candidate:
            normalized.append(candidate)
    return normalized

# Extract and process PopQA data
print("\nExtracting questions...")
popqa_data = []
for i, example in enumerate(tqdm(popqa_ds, desc="Processing PopQA")):
    # Match based on title
    wikipedia_title = example.get("s_wiki_title")
    if not wikipedia_title:
        # Fallback to subj if s_wiki_title is somehow missing
        wikipedia_title = example.get("subj")
    
    if not wikipedia_title:
        continue

    possible_answers = normalize_possible_answers(example.get("possible_answers"))
    answer_texts = possible_answers[:1]
    row = {
        "question_id": example.get("id", f"popqa_{i}"),
        "question_text": example["question"],
        "answer_texts": answer_texts,
        "wikipedia_title": wikipedia_title,
    }
    popqa_data.append(row)

popqa_df = pd.DataFrame(popqa_data)

# Merge with popularity data using Wikipedia Title
print("\nMerging with popularity data via Wikipedia Title...")
popqa_merged = popqa_df.merge(
    pop_df,
    how="inner",
    on="wikipedia_title",
 )

# Reorder columns (wikipedia_id comes from pop_df now)
popqa_merged = popqa_merged[[
    "question_id", "question_text", "answer_texts", 
    "wikipedia_id", "wikipedia_title", "popularity_avg", "popularity_rank"
 ]]

# Save
popqa_merged.to_parquet(POPQA_OUTPUT, index=False)

# Report
matched_pct = (len(popqa_merged) / len(popqa_df)) * 100
print(f"\n✓ PopQA Processing Complete:")
print(f"  Total questions: {len(popqa_df):,}")
print(f"  Matched with popularity: {len(popqa_merged):,} ({matched_pct:.1f}%)")
print(f"  Discarded: {len(popqa_df) - len(popqa_merged):,}")
print(f"  Saved to: {POPQA_OUTPUT}")
display(popqa_merged.head())

# Clean up

del popqa_ds, popqa_df
gc.collect()

PROCESSING POPQA DATASET

Loading PopQA dataset...


PROCESSING POPQA DATASET

Loading PopQA dataset...


Repo card metadata block was not found. Setting CardData to empty.


PROCESSING POPQA DATASET

Loading PopQA dataset...


Repo card metadata block was not found. Setting CardData to empty.


...

Loaded 14,267 questions

Sample question:
{'id': 4222362, 'subj': 'George Rankin', 'prop': 'occupation', 'obj': 'politician', 'subj_id': 1850297, 'prop_id': 22, 'obj_id': 2834605, 's_aliases': '["George James Rankin"]', 'o_aliases': '["political leader","political figure","polit.","pol"]', 's_uri': 'http://www.wikidata.org/entity/Q5543720', 'o_uri': 'http://www.wikidata.org/entity/Q82955', 's_wiki_title': 'George Rankin', 'o_wiki_title': 'Politician', 's_pop': 142, 'o_pop': 25692, 'question': "What is George Rankin's occupation?", 'possible_answers': '["politician", "political leader", "political figure", "polit.", "pol"]'}

Extracting questions...


Processing PopQA: 100%|██████████| 14267/14267 [00:00<00:00, 27654.27it/s]



Merging with popularity data via Wikipedia Title...

✓ PopQA Processing Complete:
  Total questions: 14,267
  Matched with popularity: 13,811 (96.8%)
  Discarded: 456
  Saved to: /Users/cyro/Documents/VSC/PopularityBias/data/popqa_with_popularity.parquet


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank
0,4222362,What is George Rankin's occupation?,[politician],16304924,George Rankin,70.770833,2.864337e+06
1,4725190,What is John Mayne's occupation?,[journalist],1098597,John Mayne,79.125000,2.780602e+06
2,4382392,What is Henry Feilden's occupation?,[politician],30108267,Henry Feilden (Conservative politician),17.895833,4.422461e+06
3,4822110,What is Kathy Saltzman's occupation?,[politician],23256856,Kathy Saltzman,40.270833,3.489112e+06
4,4011112,What is Eleanor Davis's occupation?,[cartoonist],27077682,Eleanor Davis,249.000000,1.664860e+06


0

## 4. Process Natural Questions (NQ) Dataset

In [22]:
print("="*80)
print("PROCESSING NATURAL QUESTIONS DATASET")
print("="*80)

# Load NQ from KILT
print("\nLoading Natural Questions dataset...")
nq_ds = load_dataset(
    NQ_PATH,
    name="nq",
    split="train+validation+test",
    cache_dir=CACHE_DIR
)

print(f"Loaded {len(nq_ds):,} questions")
print("\nSample question:")
print(nq_ds[0])

# Extract and process NQ data
print("\nExtracting questions...")
nq_data = []
for example in tqdm(nq_ds, desc="Processing NQ"):
    # Extract answers
    answers = []
    if "output" in example and example["output"]:
        for output in example["output"]:
            if "answer" in output:
                answers.append(output["answer"])
    
    # Extract Wikipedia IDs from provenance
    wikipedia_ids = set()
    if "output" in example and example["output"]:
        for output in example["output"]:
            if "provenance" in output:
                for prov in output["provenance"]:
                    if "wikipedia_id" in prov:
                        try:
                            wikipedia_ids.add(int(prov["wikipedia_id"]))
                        except (ValueError, TypeError):
                            continue
    
    # Create a row for each Wikipedia ID
    if wikipedia_ids:
        for wiki_id in wikipedia_ids:
            row = {
                "question_id": example["id"],
                "question_text": example["input"],
                "answer_texts": answers,
                "wikipedia_id": wiki_id,
            }
            nq_data.append(row)

nq_df = pd.DataFrame(nq_data)
if not nq_df.empty:
    nq_df["wikipedia_id"] = nq_df["wikipedia_id"].astype("int64")
print(f"\nExtracted {len(nq_df):,} question-document pairs")

# Merge with popularity data using Wikipedia ID
print("\nMerging with popularity data via Wikipedia ID...")
nq_merged = nq_df.merge(
    pop_df,
    how="inner",
    on="wikipedia_id",
    validate="m:1"
 )

# Reorder columns
nq_merged = nq_merged[[
    "question_id", "question_text", "answer_texts",
    "wikipedia_id", "wikipedia_title", "popularity_avg", "popularity_rank"
]]

# Save
nq_merged.to_parquet(NQ_OUTPUT, index=False)

# Report
matched_pct = (len(nq_merged) / len(nq_df)) * 100
print(f"\n✓ Natural Questions Processing Complete:")
print(f"  Total question-document pairs: {len(nq_df):,}")
print(f"  Matched with popularity: {len(nq_merged):,} ({matched_pct:.1f}%)")
print(f"  Discarded: {len(nq_df) - len(nq_merged):,}")
print(f"  Saved to: {NQ_OUTPUT}")
display(nq_merged.head())

# Clean up
del nq_ds, nq_df
gc.collect()

PROCESSING NATURAL QUESTIONS DATASET

Loading Natural Questions dataset...
Loaded 91,653 questions

Sample question:
{'id': '5328212470870865242', 'input': 'how i.met your mother who is the mother', 'meta': {'left_context': '', 'mention': '', 'right_context': '', 'partial_evidence': [], 'obj_surface': [], 'sub_surface': [], 'subj_aliases': [], 'template_questions': []}, 'output': [{'answer': 'Tracy McConnell', 'meta': {'score': -1}, 'provenance': [{'bleu_score': 1.0, 'start_character': 0, 'start_paragraph_id': 1, 'end_character': 15, 'end_paragraph_id': 1, 'meta': {'fever_page_id': '', 'fever_sentence_id': -1, 'annotation_id': '-1', 'yes_no_answer': '', 'evidence_span': []}, 'section': 'Section::::Abstract.', 'title': 'The Mother (How I Met Your Mother)', 'wikipedia_id': '40262098'}]}, {'answer': '', 'meta': {'score': -1}, 'provenance': [{'bleu_score': 0.8409682512283325, 'start_character': 41, 'start_paragraph_id': 1, 'end_character': 487, 'end_paragraph_id': 1, 'meta': {'fever_page_i

PROCESSING NATURAL QUESTIONS DATASET

Loading Natural Questions dataset...
Loaded 91,653 questions

Sample question:
{'id': '5328212470870865242', 'input': 'how i.met your mother who is the mother', 'meta': {'left_context': '', 'mention': '', 'right_context': '', 'partial_evidence': [], 'obj_surface': [], 'sub_surface': [], 'subj_aliases': [], 'template_questions': []}, 'output': [{'answer': 'Tracy McConnell', 'meta': {'score': -1}, 'provenance': [{'bleu_score': 1.0, 'start_character': 0, 'start_paragraph_id': 1, 'end_character': 15, 'end_paragraph_id': 1, 'meta': {'fever_page_id': '', 'fever_sentence_id': -1, 'annotation_id': '-1', 'yes_no_answer': '', 'evidence_span': []}, 'section': 'Section::::Abstract.', 'title': 'The Mother (How I Met Your Mother)', 'wikipedia_id': '40262098'}]}, {'answer': '', 'meta': {'score': -1}, 'provenance': [{'bleu_score': 0.8409682512283325, 'start_character': 41, 'start_paragraph_id': 1, 'end_character': 487, 'end_paragraph_id': 1, 'meta': {'fever_page_i

Processing NQ: 100%|██████████| 91653/91653 [00:03<00:00, 24222.08it/s]


PROCESSING NATURAL QUESTIONS DATASET

Loading Natural Questions dataset...
Loaded 91,653 questions

Sample question:
{'id': '5328212470870865242', 'input': 'how i.met your mother who is the mother', 'meta': {'left_context': '', 'mention': '', 'right_context': '', 'partial_evidence': [], 'obj_surface': [], 'sub_surface': [], 'subj_aliases': [], 'template_questions': []}, 'output': [{'answer': 'Tracy McConnell', 'meta': {'score': -1}, 'provenance': [{'bleu_score': 1.0, 'start_character': 0, 'start_paragraph_id': 1, 'end_character': 15, 'end_paragraph_id': 1, 'meta': {'fever_page_id': '', 'fever_sentence_id': -1, 'annotation_id': '-1', 'yes_no_answer': '', 'evidence_span': []}, 'section': 'Section::::Abstract.', 'title': 'The Mother (How I Met Your Mother)', 'wikipedia_id': '40262098'}]}, {'answer': '', 'meta': {'score': -1}, 'provenance': [{'bleu_score': 0.8409682512283325, 'start_character': 41, 'start_paragraph_id': 1, 'end_character': 487, 'end_paragraph_id': 1, 'meta': {'fever_page_i

Processing NQ: 100%|██████████| 91653/91653 [00:03<00:00, 24222.08it/s]


...

PROCESSING NATURAL QUESTIONS DATASET

Loading Natural Questions dataset...
Loaded 91,653 questions

Sample question:
{'id': '5328212470870865242', 'input': 'how i.met your mother who is the mother', 'meta': {'left_context': '', 'mention': '', 'right_context': '', 'partial_evidence': [], 'obj_surface': [], 'sub_surface': [], 'subj_aliases': [], 'template_questions': []}, 'output': [{'answer': 'Tracy McConnell', 'meta': {'score': -1}, 'provenance': [{'bleu_score': 1.0, 'start_character': 0, 'start_paragraph_id': 1, 'end_character': 15, 'end_paragraph_id': 1, 'meta': {'fever_page_id': '', 'fever_sentence_id': -1, 'annotation_id': '-1', 'yes_no_answer': '', 'evidence_span': []}, 'section': 'Section::::Abstract.', 'title': 'The Mother (How I Met Your Mother)', 'wikipedia_id': '40262098'}]}, {'answer': '', 'meta': {'score': -1}, 'provenance': [{'bleu_score': 0.8409682512283325, 'start_character': 41, 'start_paragraph_id': 1, 'end_character': 487, 'end_paragraph_id': 1, 'meta': {'fever_page_i

Processing NQ: 100%|██████████| 91653/91653 [00:03<00:00, 24222.08it/s]



Extracted 81,561 question-document pairs

Merging with popularity data via Wikipedia ID...

✓ Natural Questions Processing Complete:
  Total question-document pairs: 81,561
  Matched with popularity: 81,533 (100.0%)
  Discarded: 28
  Saved to: /Users/cyro/Documents/VSC/PopularityBias/data/nq_with_popularity.parquet


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank
0,5328212470870865242,how i.met your mother who is the mother,"[Tracy McConnell, ]",40262098,The Mother (How I Met Your Mother),34949.312500,31208.906250
1,5289242154789678439,who had the most wins in the nfl,"[Tom Brady, ]",13929036,List of National Football League career quarte...,19247.145833,94800.020833
2,-2500044561429484630,who played mantis guardians of the galaxy 2,"[Pom Klementieff, ]",43209054,Pom Klementieff,110204.416667,10183.697917
3,-7491001389340565191,god's not dead a light in the darkness release...,"[March 30 , 2018, ]",56095752,God's Not Dead: A Light in Darkness,7215.583333,167099.197917
4,4706363175863681196,when do the eclipse supposed to take place,"[August 21 , 2017]",4762233,"Solar eclipse of August 21, 2017",7988.354167,156981.906250


0

## 5. Process TriviaQA Dataset

In [23]:
print("="*80)
print("PROCESSING TRIVIAQA DATASET")
print("="*80)

# Load TriviaQA from KILT
print("\nLoading TriviaQA dataset...")
triviaqa_kilt = load_dataset(
    TRIVIAQA_PATH,
    name="triviaqa_support_only",
    split="train+validation+test",
    cache_dir=CACHE_DIR
)

print(f"Loaded {len(triviaqa_kilt):,} KILT TriviaQA questions")

# Load original TriviaQA to get question text
print("\nLoading original TriviaQA dataset for question text...")
trivia_qa = load_dataset(
    'trivia_qa',
    'unfiltered.nocontext',
    split="train+validation+test",
    cache_dir=CACHE_DIR
)

print(f"Loaded {len(trivia_qa):,} original TriviaQA questions")

# Create mapping from KILT ID to TriviaQA index
print("\nMapping KILT IDs to TriviaQA questions...")
triviaqa_map = dict([(q_id, i) for i, q_id in enumerate(trivia_qa['question_id'])])

# Extract and process TriviaQA data
print("\nExtracting questions...")
triviaqa_data = []
skipped = 0

for example in tqdm(triviaqa_kilt, desc="Processing TriviaQA"):
    kilt_id = example["id"]
    
    # Skip if ID not in mapping
    if kilt_id not in triviaqa_map:
        skipped += 1
        continue
    
    # Get original question data
    trivia_idx = triviaqa_map[kilt_id]
    question_text = trivia_qa[trivia_idx]['question']
    answer_value = trivia_qa[trivia_idx]['answer']['value']
    
    # Extract answers from KILT
    answers = [answer_value]
    if "output" in example and example["output"]:
        for output in example["output"]:
            if "answer" in output and output["answer"] not in answers:
                answers.append(output["answer"])
    
    # Extract Wikipedia IDs from provenance
    wikipedia_ids = set()
    if "output" in example and example["output"]:
        for output in example["output"]:
            if "provenance" in output:
                for prov in output["provenance"]:
                    if "wikipedia_id" in prov:
                        try:
                            wikipedia_ids.add(int(prov["wikipedia_id"]))
                        except (ValueError, TypeError):
                            continue
    
    # Create a row for each Wikipedia ID
    if wikipedia_ids:
        for wiki_id in wikipedia_ids:
            row = {
                "question_id": kilt_id,
                "question_text": question_text,
                "answer_texts": answers,
                "wikipedia_id": wiki_id,
            }
            triviaqa_data.append(row)

print(f"\nSkipped {skipped:,} questions without ID mapping")
triviaqa_df = pd.DataFrame(triviaqa_data)
if not triviaqa_df.empty:
    triviaqa_df["wikipedia_id"] = triviaqa_df["wikipedia_id"].astype("int64")
print(f"Extracted {len(triviaqa_df):,} question-document pairs")

# Merge with popularity data using Wikipedia ID
print("\nMerging with popularity data via Wikipedia ID...")
triviaqa_merged = triviaqa_df.merge(
    pop_df,
    how="inner",
    on="wikipedia_id",
    validate="m:1"
 )

# Reorder columns
triviaqa_merged = triviaqa_merged[[
    "question_id", "question_text", "answer_texts",
    "wikipedia_id", "wikipedia_title", "popularity_avg", "popularity_rank"
]]

# Save
triviaqa_merged.to_parquet(TRIVIAQA_OUTPUT, index=False)

# Report
matched_pct = (len(triviaqa_merged) / len(triviaqa_df)) * 100 if len(triviaqa_df) > 0 else 0
print(f"\n✓ TriviaQA Processing Complete:")
print(f"  Total question-document pairs: {len(triviaqa_df):,}")
print(f"  Matched with popularity: {len(triviaqa_merged):,} ({matched_pct:.1f}%)")
print(f"  Discarded: {len(triviaqa_df) - len(triviaqa_merged):,}")
print(f"  Saved to: {TRIVIAQA_OUTPUT}")
display(triviaqa_merged.head())

# Clean up
del triviaqa_kilt, trivia_qa, triviaqa_df
gc.collect()

PROCESSING TRIVIAQA DATASET

Loading TriviaQA dataset...
Loaded 73,789 KILT TriviaQA questions

Loading original TriviaQA dataset for question text...


PROCESSING TRIVIAQA DATASET

Loading TriviaQA dataset...
Loaded 73,789 KILT TriviaQA questions

Loading original TriviaQA dataset for question text...


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

PROCESSING TRIVIAQA DATASET

Loading TriviaQA dataset...
Loaded 73,789 KILT TriviaQA questions

Loading original TriviaQA dataset for question text...


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

...

Loaded 109,767 original TriviaQA questions

Mapping KILT IDs to TriviaQA questions...

Extracting questions...


Processing TriviaQA: 100%|██████████| 73789/73789 [00:16<00:00, 4366.34it/s]



Skipped 39 questions without ID mapping
Extracted 98,400 question-document pairs

Merging with popularity data via Wikipedia ID...

✓ TriviaQA Processing Complete:
  Total question-document pairs: 98,400
  Matched with popularity: 98,386 (100.0%)
  Discarded: 14
  Saved to: /Users/cyro/Documents/VSC/PopularityBias/data/triviaqa_with_popularity.parquet


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank
0,dpql_5197,What denomination of British bank note depicts...,"[£5, five £, 5 £, five £]",270680,Banknotes of the pound sterling,20593.020833,55316.343750
1,dpql_5199,"Who connects the BBC news, ‘Crimewatch’, and ‘...","[FIONA BRUCE, Fiona Bruce, fiona bruce]",23297505,Crimewatch,5011.708333,224056.541667
2,dpql_5199,"Who connects the BBC news, ‘Crimewatch’, and ‘...","[FIONA BRUCE, Fiona Bruce, fiona bruce]",538901,Antiques Roadshow,11162.229167,103070.781250
3,dpql_5199,"Who connects the BBC news, ‘Crimewatch’, and ‘...","[FIONA BRUCE, Fiona Bruce, fiona bruce]",1139893,BBC News,58191.958333,15076.427083
4,dpql_5200,"Born in 1963, whose real name is Georgios Kyri...","[GEORGE MICHAEL, Georgios Panayiotou, Anselmo ...",45985,George Michael,355779.312500,940.625000


0

## 6. Summary Statistics

In [24]:
print("="*80)
print("FINAL SUMMARY")
print("="*80)

# Load all generated files
datasets = {
    "PopQA": POPQA_OUTPUT,
    "Natural Questions": NQ_OUTPUT,
    "TriviaQA": TRIVIAQA_OUTPUT
}

print("\nGenerated Files:")
for name, path in datasets.items():
    if path.exists():
        df = pd.read_parquet(path)
        print(f"\n{name}:")
        print(f"  File: {path}")
        print(f"  Rows: {len(df):,}")
        print(f"  Unique questions: {df['question_id'].nunique():,}")
        print(f"  Unique Wikipedia pages: {df['wikipedia_id'].nunique():,}")
        print(f"  Columns: {', '.join(df.columns)}")
        print(f"  Popularity range: {df['popularity_avg'].min():.2f} - {df['popularity_avg'].max():.2f}")
        print(f"  Median popularity: {df['popularity_avg'].median():.2f}")
    else:
        print(f"\n{name}: ❌ File not generated")

print("\n" + "="*80)
print("ALL DATASETS PROCESSED SUCCESSFULLY!")
print("="*80)

FINAL SUMMARY

Generated Files:

PopQA:
  File: /Users/cyro/Documents/VSC/PopularityBias/data/popqa_with_popularity.parquet
  Rows: 13,811
  Unique questions: 13,811
  Unique Wikipedia pages: 11,852
  Columns: question_id, question_text, answer_texts, wikipedia_id, wikipedia_title, popularity_avg, popularity_rank
  Popularity range: 1.00 - 2263275.23
  Median popularity: 903.02

Natural Questions:
  File: /Users/cyro/Documents/VSC/PopularityBias/data/nq_with_popularity.parquet
  Rows: 81,533
  Unique questions: 79,756
  Unique Wikipedia pages: 39,402
  Columns: question_id, question_text, answer_texts, wikipedia_id, wikipedia_title, popularity_avg, popularity_rank
  Popularity range: 1.00 - 7144786.38
  Median popularity: 18366.56

TriviaQA:
  File: /Users/cyro/Documents/VSC/PopularityBias/data/triviaqa_with_popularity.parquet
  Rows: 98,386
  Unique questions: 58,215
  Unique Wikipedia pages: 38,389
  Columns: question_id, question_text, answer_texts, wikipedia_id, wikipedia_title, po

## 7. Upload Enriched QA Files to HuggingFace

Authenticate via `HUGGINGFACE_TOKEN` and push each parquet artifact to the configured repo so downstream consumers can access the enriched QA datasets.

In [26]:
from huggingface_hub import login, HfApi
from datasets import Dataset, DatasetDict
from io import BytesIO
import pandas as pd

if not HUGGINGFACE_TOKEN:
    raise RuntimeError("HUGGINGFACE_TOKEN is not set; cannot upload datasets.")

print("Logging in to HuggingFace...")
login(token=HUGGINGFACE_TOKEN)

# Upload README
print("Uploading README.md...")
api = HfApi()
api.upload_file(
    path_or_fileobj=BytesIO(README_TEXT.strip().encode("utf-8")),
    path_in_repo="README.md",
    repo_id=HUGGINGFACE_UPLOAD_REPO,
    repo_type="dataset",
    token=HUGGINGFACE_TOKEN,
)

for label, (subset_name, dataset_path) in HUGGINGFACE_UPLOAD_SUBSETS.items():
    if not dataset_path.exists():
        print(f"Skipping {label}: File not found.")
        continue

    print(f"Processing {subset_name}...")
    df = pd.read_parquet(dataset_path)
    df["question_id"] = df["question_id"].astype(str)
    
    # 90/10 Train/Test split
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    split_idx = int(len(df) * 0.9)
    train_df, test_df = df.iloc[:split_idx], df.iloc[split_idx:]
    
    ds_dict = DatasetDict({
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "test": Dataset.from_pandas(test_df, preserve_index=False)
    })
    
    ds_dict.push_to_hub(
        repo_id=HUGGINGFACE_UPLOAD_REPO,
        config_name=subset_name,
        token=HUGGINGFACE_TOKEN
    )
    print(f"✓ Pushed {subset_name} configuration")

print(f"\nUpload complete: https://huggingface.co/datasets/{HUGGINGFACE_UPLOAD_REPO}")

Logging in to HuggingFace...
Uploading README.md...
Processing popqa...


Logging in to HuggingFace...
Uploading README.md...
Processing popqa...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/13 [00:00<?, ?ba/s]

Logging in to HuggingFace...
Uploading README.md...
Processing popqa...


...

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

✓ Pushed triviaqa configuration

Upload complete: https://huggingface.co/datasets/Cyro1/popularity-enriched-qa-datasets
